In [ ]:
!pip install --upgrade pip
!pip install pandas==3.0.2

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

np.random.seed(42)
N = 2000
names = ['Иван', 'Ольга', 'Дмитрий', 'Екатерина', 'Сергей', 'Анна', 'Алексей', 'Мария', 'Николай', 'Татьяна']

df = pd.DataFrame({
    'client_id': range(1001, 1001 + N),
    'name': np.random.choice(names, N),
    'age': np.random.randint(21, 66, N),
    'income': np.random.randint(15000, 500001, N),
    'dti': np.round(np.random.uniform(0, 1, N), 2),
    'default': np.random.choice([True, False], N, p=[0.12, 0.88]),
    'application_date': pd.date_range('2024-01-01', periods=N, freq='h')
})

df.set_index('client_id', inplace=True)
print(f"Датасет: {df.shape[0]} строк, {df.shape[1]} колонок")
df.head()

Pandas version: 3.0.2
Датасет: 2000 строк, 6 колонок


,name,age,income,dti,default,application_date
client_id,,,,,,
1001,Алексей,62,162595,0.97,False,2024-01-01 00:00:00
1002,Екатерина,59,344104,0.05,True,2024-01-01 01:00:00
1003,Мария,61,263709,0.89,False,2024-01-01 02:00:00
1004,Сергей,31,296696,0.58,False,2024-01-01 03:00:00
1005,Алексей,59,156880,0.56,False,2024-01-01 04:00:00


## Простая булева маска

Найди всех клиентов младше 30 лет.

In [ ]:
young_mask = df['age'] < 30
young_clients = df[young_mask]
print(f"Клиентов младше 30: {len(young_clients)}")
print(young_clients.head())

Клиентов младше 30: 390
              name  age  income   dti  default    application_date
client_id                                                         
1008       Алексей   24  169692  0.09    False 2024-01-01 07:00:00
1020         Ольга   23  354933  0.68    False 2024-01-01 19:00:00
1021        Сергей   29   99526  0.68    False 2024-01-01 20:00:00
1022          Иван   28  416062  0.95    False 2024-01-01 21:00:00
1024          Анна   26   61220  0.00    False 2024-01-01 23:00:00


## Комбинация условий (&)

Найди клиентов с доходом от 50 000 до 100 000 включительно и DTI меньше 0.5.

In [ ]:
condition = (df['income'] >= 50000) & (df['income'] <= 100000) & (df['dti'] < 0.5)
result = df[condition]
print(f"Подошло клиентов: {len(result)}")
print(result.head())

Подошло клиентов: 96
                name  age  income   dti  default    application_date
client_id                                                           
1015            Анна   56   96938  0.29     True 2024-01-01 14:00:00
1024            Анна   26   61220  0.00    False 2024-01-01 23:00:00
1030       Екатерина   25   90277  0.02     True 2024-01-02 05:00:00
1041         Николай   42   66913  0.10    False 2024-01-02 16:00:00
1064         Татьяна   36   74778  0.10    False 2024-01-03 15:00:00


## Использование | (ИЛИ)
Найди клиентов, у которых доход меньше 20 000 ИЛИ DTI больше 0.8.

In [ ]:
low_income_or_high_dti = df[(df['income'] < 20000) | (df['dti'] > 0.8)]
print(f"Клиентов с низким доходом или высоким DTI: {len(low_income_or_high_dti)}")

Клиентов с низким доходом или высоким DTI: 411


## Отрицание ~
Найди всех клиентов, НЕ являющихся Иванами.

In [ ]:
not_ivan = df[~(df['name'] == 'Иван')]
print(f"Не Иванов: {len(not_ivan)} из {len(df)}")

Не Иванов: 1776 из 2000


## Метод query()


In [ ]:
result_query = df.query('income >= 50000 & income <= 100000 & dti < 0.5')
print(f"Подошло клиентов (query): {len(result_query)}")

Подошло клиентов (query): 96


## between()
Найди клиентов с возрастом от 35 до 50 лет (включительно) через between().

In [ ]:
age_between = df[df['age'].between(35, 50)]
print(f"Клиентов 35–50 лет: {len(age_between)}")

Клиентов 35–50 лет: 728


## isin()
Найди клиентов с именами 'Иван', 'Мария', 'Алексей'.

In [ ]:
selected_names = df[df['name'].isin(['Иван', 'Мария', 'Алексей'])]
print(f"Клиентов с именами Иван, Мария, Алексей: {len(selected_names)}")

Клиентов с именами Иван, Мария, Алексей: 617


## str.startswith()
Найди клиентов, чьё имя начинается на букву 'А'.

In [ ]:
starts_with_A = df[df['name'].str.startswith('А')]
print(f"Имена на 'А': {len(starts_with_A)}")
print(starts_with_A['name'].value_counts())

Имена на 'А': 384
name
Алексей    193
Анна       191
Name: count, dtype: int64


## where() и mask()
- Замени доход на NaN, если он меньше 30 000 (where).
- Обнули DTI для клиентов старше 60 лет (mask).

In [ ]:
# where: оставить доход только если >= 30000, иначе NaN
df['income_clean'] = df['income'].where(df['income'] >= 30000, other=None)
print("Пропусков в income_clean:", df['income_clean'].isna().sum())

# mask: обнулить DTI для клиентов старше 60
df['dti_adjusted'] = df['dti'].mask(df['age'] > 60, 0.0)
print("Средний DTI после корректировки:", df['dti_adjusted'].mean())

Пропусков в income_clean: 61
Средний DTI после корректировки: 0.443445


##
"Нашальникэ" просит выборку для анализа кисков-рисков:
- возраст от 30 до 50 лет
- доход от 100 000 до 400 000
- DTI меньше 0.5
- нет дефолта (default == False)
- только колонки name, age, income, dti

In [ ]:
risk_analysis = df.query(
    'age.between(30, 50) & income.between(100000, 400000) & dti < 0.5 & default == False'
)[['name', 'age', 'income', 'dti']]

print(f"Отобрано клиентов: {len(risk_analysis)}")
print(risk_analysis.head(10))

Отобрано клиентов: 241
                name  age  income   dti
client_id                              
1058            Анна   33  143467  0.38
1061       Екатерина   36  233800  0.24
1075           Мария   43  150189  0.40
1097         Николай   34  180582  0.13
1114         Дмитрий   38  319281  0.25
1129         Алексей   50  264880  0.02
1130         Алексей   44  332420  0.32
1138          Сергей   34  342762  0.45
1149       Екатерина   40  351047  0.16
1160           Мария   36  274480  0.44
